In [7]:
import json, re
from datetime import datetime

# ---------- 1) 파일 열기 ----------
with open("news_2025.04.27_2025.06.03.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# ---------- 2) 날짜 파싱 ----------
date_pat = re.compile(r"\d{4}\.\d{2}\.\d{2}")

def to_dt(date_str: str) -> datetime:
    """'2025.04.27. 오후 8:46' → datetime.date(2025, 4, 27)"""
    m = date_pat.search(date_str)
    if not m:
        raise ValueError(f"날짜 파싱 실패: {date_str}")
    return datetime.strptime(m.group(), "%Y.%m.%d")

start = datetime(2025, 5, 29)
end   = datetime(2025, 6, 3)

def in_range(article: dict) -> bool:
    """삭제 대상 기간인지 여부"""
    try:
        return start <= to_dt(article["date"]) <= end
    except Exception:
        return False  # 날짜 필드가 없거나 형식이 다르면 살려둠

# ---------- 3) 재귀적으로 자료구조 필터링 ----------
def prune(obj):
    """
    obj가
      - 기사 dict ➜ 기간 안이면 None, 아니면 그대로
      - list      ➜ 내부 항목을 재귀 처리해 None/빈값 제거
      - dict      ➜ value 재귀 처리 후 None/빈값 제거
    """
    # (1) 기사 딕셔너리인가?
    if isinstance(obj, dict) and "date" in obj and "title" in obj:
        return None if in_range(obj) else obj

    # (2) 리스트이면 각 요소 재귀 처리
    if isinstance(obj, list):
        kept = [prune(x) for x in obj]
        kept = [x for x in kept if x not in (None, [], {})]
        return kept

    # (3) 딕셔너리이면 value 재귀 처리
    if isinstance(obj, dict):
        pruned = {k: prune(v) for k, v in obj.items()}
        pruned = {k: v for k, v in pruned.items() if v not in (None, [], {})}
        return pruned

    # (4) 그 외 타입은 그대로
    return obj

filtered = prune(data)

# ---------- 4) 결과 저장 ----------
with open("news_filtered.json", "w", encoding="utf-8") as f:
    json.dump(filtered, f, ensure_ascii=False, indent=2)

print("필터링 완료!  news_filtered.json 로 저장되었습니다.")

필터링 완료!  news_filtered.json 로 저장되었습니다.


In [4]:
import json 

# ---------- 1) 원본 파일 읽기 ----------
with open("news_filtered_2025.04.27_2025.05.28.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# ---------- 2) 재귀 함수: content 키 삭제 ----------
def drop_content(obj):
    """
    obj가
      - 기사 dict : 'content' 키 제거
      - list      : 내부 요소 재귀 처리
      - dict      : value 재귀 처리
    """
    # (1) 기사 딕셔너리라면 pop
    if isinstance(obj, dict) and "title" in obj and "date" in obj:
        obj.pop("content", None)  # 키가 없으면 무시
        return obj

    # (2) 리스트 -> 내부 요소 순회
    if isinstance(obj, list):
        return [drop_content(x) for x in obj]

    # (3) 일반 딕셔너리 -> value 순회
    if isinstance(obj, dict):
        return {k: drop_content(v) for k, v in obj.items()}

    # (4) 그 외 타입은 그대로
    return obj

data = drop_content(data)

# ---------- 3) 새 파일로 저장 ----------
with open("news_no_content.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("✅ 'content' 필드가 제거된 news_no_content.json 생성 완료")


✅ 'content' 필드가 제거된 news_no_content.json 생성 완료


In [1]:
import json
import itertools

# ---------- 1) 원본 파일 읽기 ----------
with open("news_no_content.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# ---------- 2) 재귀적으로 order 재부여 ----------
def renumber(obj, counter):
    """
    obj가
      - 기사 dict : 'order' 키를 새 번호로 덮어씀
      - list      : 내부 요소 재귀 처리
      - dict      : value 재귀 처리
    """
    if isinstance(obj, dict):
        # 기사로 간주할 기준: 'id'와 'order' 키가 모두 존재
        if "order" in obj and "id" in obj:
            obj["order"] = next(counter)
        # 내부 값에도 재귀 적용
        for v in obj.values():
            renumber(v, counter)

    elif isinstance(obj, list):
        for item in obj:
            renumber(item, counter)

# 카운터: 1,2,3,…
counter = itertools.count(1)
renumber(data, counter)

# ---------- 3) 새 파일로 저장 ----------
with open("news_reordered.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("✅ order 필드를 1부터 연속 번호로 갱신한 news_reordered.json 생성 완료")


✅ order 필드를 1부터 연속 번호로 갱신한 news_reordered.json 생성 완료
